<a href="https://colab.research.google.com/github/RuchirS-spec/qmlhep/blob/main/gsoc_task2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Installing dependencies ->

In [11]:
# Install PyTorch Geometric and dependencies
import torch

# Ensure torch is imported to get its version
print(f"Using torch version: {torch.__version__}")

# Install PyTorch Geometric dependencies. Removing -f flag to allow pip to find compatible wheels.
!pip install -q torch-scatter
!pip install -q torch-sparse
!pip install -q torch-cluster
!pip install -q torch-geometric
!pip install -q tqdm

Using torch version: 2.7.1+cu128


  error: subprocess-exited-with-error
  
  Getting requirements to build wheel did not run successfully.
  exit code: 1
  
  [23 lines of output]
  Traceback (most recent call last):
    File "C:\Users\ruchi\AppData\Roaming\Python\Python314\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 389, in <module>
      main()
      ~~~~^^
    File "C:\Users\ruchi\AppData\Roaming\Python\Python314\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 373, in main
      json_out["return_val"] = hook(**hook_input["kwargs"])
                               ~~~~^^^^^^^^^^^^^^^^^^^^^^^^
    File "C:\Users\ruchi\AppData\Roaming\Python\Python314\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 143, in get_requires_for_build_wheel
      return hook(config_settings)
    File "C:\Users\ruchi\AppData\Local\Temp\pip-build-env-logowngu\overlay\Lib\site-packages\setuptools\build_meta.py", line 333, in get_requires_for_build_wheel
     

In [13]:
# 1. Upgrade Torch to the 2026 standard
!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

# 2. Install PyG and its optimized backend
!pip install torch_geometric pyg_lib torch_scatter torch_sparse torch_cluster

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cu128
   ---------------------------------------- 0.0/2.8 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 GB 18.3 MB/s eta 0:02:32
   ---------------------------------------- 0.0/2.8 GB 16.7 MB/s eta 0:02:46
   ---------------------------------------- 0.0/2.8 GB 17.8 MB/s eta 0:02:36
   ---------------------------------------- 0.0/2.8 GB 18.6 MB/s eta 0:02:29
   ---------------------------------------- 0.0/2.8 GB 18.5 MB/s eta 0:02:29
   ---------------------------------------- 0.0/2.8 GB 18.6 MB/s eta 0:02:28
   ---------------------------------------- 0.0/2.8 GB 19.1 MB/s eta 0:02:24
   ---------------------------------------- 0.0/2.8 GB 18.9 MB/s eta 0:02:26
    --------------------------------------- 0.0/2.8 GB 18.9 MB/s eta 0:02:25
    --------------------------------------- 0.0/2.8 GB 18.9 MB/s eta 0:02:25
    --------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


Defaulting to user installation because normal site-packages is not writeable


ERROR: Could not find a version that satisfies the requirement pyg_lib (from versions: none)
ERROR: No matching distribution found for pyg_lib


## Data Loading ->

In [9]:
import os
import zipfile
import numpy as np
import torch
from tqdm import tqdm

ZIP_PATH = "/content/3164691.zip"
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

Using device: cuda



### 1. The k-NN Construction (Capturing the Shower)
Used **k-Nearest Neighbors (k-NN)** based on the distance $\Delta R = \sqrt{\Delta \eta^2 + \Delta \phi^2}$

* **Physical Meaning:** Particles that are close together in $(\eta, \phi)$ likely originated from the same "branch" of the particle shower.
* **Local Interactions:** Connecting each node to its $k$ closest neighbors (we chose $k=16$) allows the GraphSAGE or GAT layers to perform **message passing**. A node "listens" to its neighbors to understand if it's part of a tight, narrow quark jet or a wide, messy gluon jet.

### 2. Translation and Rotation Invariance
A major innovation in this pipeline is the **Coordinate Centering**. We don't use raw $\eta$ and $\phi$ values; we use relative ones:

$$\eta_{rel} = \eta_i - \eta_{jet}$$
$$\phi_{rel} = \phi_i - \phi_{jet}$$

>  This makes the model **Translation Invariant**.

### 3. Feature Engineering: $\log(p_T)$ Scaling
In jet physics, the transverse momentum ($p_T$) of particles can span several orders of magnitude.



---

In [28]:
import os
import zipfile
import numpy as np
import torch
import random
from torch_geometric.data import Data
from tqdm import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def manual_knn(pos, k=16):
    """GPU-accelerated k-NN without torch-cluster.
    Dynamically adjusts k to prevent 'selected index k out of range' errors.
    """
    num_particles = pos.size(0)
    if num_particles <= 1:
        # No edges can be formed for 0 or 1 particle
        return torch.empty((2, 0), dtype=torch.long, device=pos.device)

    # Ensure k_actual does not exceed the number of other particles available
    k_actual = min(k, num_particles - 1)
    if k_actual == 0: # If k_actual becomes 0 after adjustment, no edges can be formed
        return torch.empty((2, 0), dtype=torch.long, device=pos.device)

    dist = torch.cdist(pos, pos)
    # Get k_actual + 1 to include the particle itself, then remove it later
    _, indices = dist.topk(k_actual + 1, largest=False)

    row = torch.arange(num_particles, device=pos.device).repeat_interleave(k_actual)
    # Exclude the first column (self-loop) and reshape to get target nodes
    col = indices[:, 1:].reshape(-1)
    return torch.stack([row, col], dim=0)

def create_pyg_data(x_jet, y_label, k=16):
    """Processes a single jet into a PyG graph."""
    mask = x_jet[:, 0] > 0
    particles = x_jet[mask]
    if len(particles) < 2: return None

    # Features and relative coordinates
    pt, eta, phi, pid = particles[:, 0], particles[:, 1], particles[:, 2], particles[:, 3]
    eta_rel = torch.tensor(eta - np.mean(eta), dtype=torch.float, device=DEVICE)
    phi_rel = torch.tensor(phi - np.mean(phi), dtype=torch.float, device=DEVICE)
    pt_log = torch.tensor(np.log(pt), dtype=torch.float, device=DEVICE)

    x = torch.stack([pt_log, eta_rel, phi_rel, torch.tensor(pid, dtype=torch.float, device=DEVICE)], dim=1)
    pos = torch.stack([eta_rel, phi_rel], dim=1)
    edge_index = manual_knn(pos, k)

    return Data(x=x, edge_index=edge_index, y=torch.tensor([y_label], dtype=torch.long, device=DEVICE))

def load_random_sample(zip_path, total_target=5000):
    dataset = []
    with zipfile.ZipFile(zip_path, 'r') as z:
        # Get and sort the 20 standard files
        all_files = z.namelist()
        qg_files = sorted([f for f in all_files if f.startswith('QG_jets') and 'withbc' not in f and f.endswith('.npz')])

        # Calculate samples per file (e.g., 5000 / 20 = 250)
        samples_per_file = total_target // len(qg_files)
        print(f"Sampling {samples_per_file} jets from each of the {len(qg_files)} files...")

        for file_name in qg_files:
            with z.open(file_name) as f:
                data = np.load(f)
                X, y = data['X'], data['y']

                # Pick random indices from this specific file
                indices = np.random.choice(len(X), samples_per_file, replace=False)

                for idx in tqdm(indices, desc=f"Sampling {file_name}", leave=False):
                    g = create_pyg_data(X[idx], y[idx])
                    if g: dataset.append(g.cpu()) # Move to CPU for storage

    return dataset

# --- Execute Load ---
ZIP_PATH = r"C:\Users\ruchi\Downloads\3164691.zip"
full_data = load_random_sample(ZIP_PATH, total_target=5000)
print(f"\nTotal Dataset Size: {len(full_data)} graphs.")

Sampling 250 jets from each of the 20 files...



Total Dataset Size: 5000 graphs.


## GNN model architectures ->

In [25]:
from torch_geometric.nn import SAGEConv, GATv2Conv, global_mean_pool
import torch.nn.functional as F

class GraphSAGEModel(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.lin = torch.nn.Linear(hidden_channels, 2)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index).relu()
        x = global_mean_pool(x, batch)
        return self.lin(x)

class GATModel(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, heads=4):
        super().__init__()
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=heads)
        self.conv2 = GATv2Conv(hidden_channels * heads, hidden_channels, heads=1)
        self.lin = torch.nn.Linear(hidden_channels, 2)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index).relu()
        x = global_mean_pool(x, batch)
        return self.lin(x)

## GNN training and benchmark

In [26]:
#train, test and eval functions->
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split

def train_and_test(model, train_loader, test_loader, epochs=10):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0
        for data in train_loader:
            data = data.to(DEVICE)
            optimizer.zero_grad()
            out = model(data)
            loss = F.cross_entropy(out, data.y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if epoch % 5 == 0:
            print(f"Epoch {epoch:02d} | Loss: {total_loss/len(train_loader):.4f}")


    model.eval()
    y_true, y_probs = [], []
    with torch.no_grad():
        for data in test_loader:
            data = data.to(DEVICE)
            out = model(data)
            y_probs.extend(F.softmax(out, dim=1)[:, 1].cpu().numpy())
            y_true.extend(data.y.cpu().numpy())

    return accuracy_score(y_true, np.round(y_probs)), roc_auc_score(y_true, y_probs)




--- Training GraphSAGE ---
Epoch 05 | Loss: 0.7033
Epoch 10 | Loss: 0.8254

--- Training GAT ---
Epoch 05 | Loss: 0.7091
Epoch 10 | Loss: 0.6902

Final Results:
GraphSAGE: Acc 0.6370, AUC 0.7091
GAT:       Acc 0.5630, AUC 0.6465


In [27]:
train_data, test_data = train_test_split(full_data, test_size=0.2, random_state=42)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64)

print("\n--- Training GraphSAGE ---")
acc_sage, auc_sage = train_and_test(GraphSAGEModel(4, 64), train_loader, test_loader)

print("\n--- Training GAT ---")
acc_gat, auc_gat = train_and_test(GATModel(4, 32), train_loader, test_loader)

print(f"\nFinal Results:\nGraphSAGE: Acc {acc_sage:.4f}, AUC {auc_sage:.4f}")
print(f"GAT:       Acc {acc_gat:.4f}, AUC {auc_gat:.4f}")


--- Training GraphSAGE ---
Epoch 05 | Loss: 0.7250
Epoch 10 | Loss: 0.7115

--- Training GAT ---
Epoch 05 | Loss: 0.7032
Epoch 10 | Loss: 0.7351

Final Results:
GraphSAGE: Acc 0.6470, AUC 0.7029
GAT:       Acc 0.5660, AUC 0.6015
